# 02 - Baseline Classification (Decision Tree)
Notebook này huấn luyện mô hình Decision Tree làm thuật toán cơ sở để so sánh với Random Forest và XGBoost.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

SEED = 42
EVENTS_PATH = '../data/processed/events_cleaned.csv'
MODEL_OUTPUT_PATH = '../models/decision_tree_baseline.pkl'
METRICS_OUTPUT_PATH = '../models/decision_tree_baseline_metrics.json'

In [ ]:
events_df = pd.read_csv(EVENTS_PATH)
events_df['timestamp'] = pd.to_datetime(events_df['timestamp'])

print('Da tai du lieu thanh cong!')
print(f'So dong: {len(events_df):,}')
events_df.head()

## Trích xuất đặc trưng theo visitor
- event_*: số lần thực hiện từng hành vi
- total_events, unique_items_viewed, unique_categories_viewed
- session_duration_seconds, most_frequent_hour
- Nhãn `converted`: 1 nếu visitor có transaction, ngược lại là 0

In [ ]:
event_features = events_df.pivot_table(
    index='visitorid',
    columns='event',
    aggfunc='size',
    fill_value=0
)
event_features.columns = ['event_' + c for c in event_features.columns]

visitor_agg_features = events_df.groupby('visitorid').agg(
    total_events=('event', 'count'),
    unique_items_viewed=('itemid', 'nunique'),
    unique_categories_viewed=('categoryid', 'nunique')
)

time_features = events_df.groupby('visitorid')['timestamp'].agg(['min', 'max'])
time_features['session_duration_seconds'] = (time_features['max'] - time_features['min']).dt.total_seconds()

events_df['hour_of_day'] = events_df['timestamp'].dt.hour
hour_features = events_df.groupby('visitorid')['hour_of_day'].agg(
    most_frequent_hour=lambda x: x.mode()[0]
)

features_df = (
    event_features
    .join(visitor_agg_features)
    .join(time_features[['session_duration_seconds']])
    .join(hour_features)
    .fillna(0)
)

if 'event_transaction' not in features_df.columns:
    features_df['event_transaction'] = 0

features_df['converted'] = (features_df['event_transaction'] > 0).astype(int)

print(f'Tong so visitor: {features_df.shape[0]:,}')
print(f"Ti le converted: {features_df['converted'].mean():.2%}")
features_df.head()

In [ ]:
y = features_df['converted']
X = features_df.drop(columns=['converted', 'event_transaction'], errors='ignore')

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'Ti le converted train: {y_train.mean():.2%}')
print(f'Ti le converted test: {y_test.mean():.2%}')

## Huấn luyện Decision Tree Baseline
Cấu hình `class_weight='balanced'` để giảm ảnh hưởng mất cân bằng lớp.

In [ ]:
dt_model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=8,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=SEED
)

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
y_proba = dt_model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy': float(accuracy_score(y_test, y_pred)),
    'precision': float(precision_score(y_test, y_pred, zero_division=0)),
    'recall': float(recall_score(y_test, y_pred, zero_division=0)),
    'f1': float(f1_score(y_test, y_pred, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, y_proba))
}

print('Decision Tree Baseline Metrics:')
for k, v in metrics.items():
    print(f'- {k}: {v:.4f}')

print('\nClassification report:')
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0])
axes[0].set_title('Confusion Matrix - Decision Tree')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title('ROC Curve - Decision Tree')

plt.tight_layout()
plt.show()

In [ ]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 15 feature importance:')
display(importance_df.head(15))

In [ ]:
import joblib

os.makedirs('../models', exist_ok=True)
joblib.dump(dt_model, MODEL_OUTPUT_PATH)

with open(METRICS_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print(f'Da luu model tai: {MODEL_OUTPUT_PATH}')
print(f'Da luu metrics tai: {METRICS_OUTPUT_PATH}')